# Lifecycle

This chapter is about the lifecycle of stream processing with Kafi Streams. It is split into three parts. 

First, we explain how to [set up and build](#setup_build) a processing topology, i.e., how to define the *topology* including *sources* and *sinks* and then how to build it.

Second, we describe the options how to [run and debug](#run_debug) the processing topology both using just the *TopologyNode* class (aka the *test driver*) and the *Streams* class (connects to Kafka).

The third and last section of this chapter is about how to [stop](#stop) the processing.


## Overview

[Preparation](#prep)

* [1 Set up and build](#setup_build)
  * [Set up](#setup)
    * [Sources](#sources)
      * [source()](#source)
        * [TopologyNode.source()](#topologynode_source)
        * [Streams.source()](#streams_source)
      * [to_zSet()](#to_zSet)
        * [from_records](#from_records)
        * [from_debezium](#from_debezium)
        * [_from_records](#_from_records)
    * [Sinks](#sinks)
        * [sink()](#sink)
          * [TopologyNode.sink()](#topologynode_sink)
          * [Streams.sink()](#streams_sink)
      * [from_zSet()](#from_zSet)
        * [to_records](#to_records)
        * [to_debezium](#to_debezium)
        * [_to_records](#_to_records)
    * [Build](#build)
      * [build()](#build)
      * [reset()](#reset)
    * [Keyword arguments](#kwargs)
      * [pack_fun()](#pack_fun)
      * [unpack_fun()](#unpack_fun)
      * [to_zSet_fun()](#to_zset_fun)
      * [from_zSet_fun()](#from_zset_fun)
* [2 Run and debug](#run_debug)
  * [Run](#run)
    * [TopologyNode (aka test driver)](#topologynode)
      * [push()](#push)
      * [latest()](#latest)
      * [process()](#process())
    * [Streams](#streams)
      * [start_streams()](#start_streams)
      * [streams()](#streams)
      * [streams_fun()](#streams_fun)
      * [threads()](#threads)
  * [Debug](#debug)
    * [Visualization](#visualization)
      * [topology()](#topology)
      * [mermaid()](#mermaid)
    * [Peeking](#peek)
    * [step_fun (Streams)](#step_fun)
* [3 Stop](#stop)
  * [stop_fun()](#stop_fun)


---
<a id="prep"></a>
## Preparation

Before we start off, we first prepare for the examples to follow:

In [1]:
!pip install -r ../requirements.txt

import sys
sys.path.insert(1, ".")
sys.path.insert(1, "../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import ClickGenerator
click_generator = ClickGenerator()
debezium_click_generator = ClickGenerator(debezium_bool=True)
weights_click_generator = ClickGenerator(weights_bool=True)

click_source_str = "clicks"
sink_str = "sink"



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] Datei oder Verzeichnis nicht gefunden: '../requirements.txt'


---
<a id="setup_build"></a>
## 1 Set up and build

Stream processing with Kafi Streams starts with setting up a topology as e.g. shown in the [Quickstart](#quickstart.ipynb).

In broad strokes, a topology consists of three parts:
1. [Sources](#sources)
2. The stream processing logic using the relational [operators](operators.ipynb) of Kafi Streams
3. [Sinks](#sinks)


<a id="sources"></a>
### Sources

You specify sources using the [`source()`](#source) operator.

With the [`to_ZSet`](#to_zSet) operator, you can configure 1) what kind of input stream Kafi Streams expects, and 2) how it converts this input stream into pydbsp's *Z-sets*.


<a id="source"></a>
#### source()

The `source()` operator differs slightly depending on whether you use the *TopologyNode* or the *Streams* class.

<a id="topologynode_source"></a>
##### TopologyNode.source()

When you use the *TopologyNode* class, the `source()` operator just takes one parameter - the name of the source topology node:

```
@staticmethod
def source(source_str, **kwargs):
"""Create a named input source node.

Args:
    source_str: name of the input source
    **kwargs: passed through to the underlying node(s)
Returns:
    tn: the newly created source topology node"""
```


<a id="streams_source"></a>
##### Streams.source()

When you use the *Streams* class, the `source()` operator takes three parameters:
1. the Kafi *Storage* (e.g. a connection to Kafka cluster) `storage`,
2. the name of the source `source_str`,
3. the name of the topic `topic_str` (optional; defaults to `source_str`)

```
@staticmethod
def source(storage, source_str, topic_str=None, **kwargs):
    """Create a source node backed by a storage topic.

    Args:
        storage: storage backend implementing producer()/consumer() (e.g. Kafka)
        source_str: name of the input source
        topic_str: topic name on storage; defaults to source_str
        **kwargs: passed to storage.consumer() at runtime
    Returns:
        tn: the newly created source topology node"""
```


<a id="to_zset"></a>
#### to_zSet()

The `to_zSet()` operator allows you to specify what kind of input stream Kafi Streams expects for the source and how this input stream is converted into *Z-sets*.

Currently, Kafi Streams offers three options:
* `from_records` This is the default - expects a list of records and converts it into a Z-set where each record gets the weight *1*. 
* `from_debezium` Expects a list of records in *Debezium* format and converts it into the corresponding Z-set.
* `_from_records` Expects a list of pairs of a record and a weight and converts into the corresponding Z-set (only makes sense for *TopologyNode*, not for *Streams*)


<a id="from_records"></a>
##### from_records

This is the default option. Lists of records, typically Kafka messages, come in and each of them gets weight `1`.

Here is an example where you can see that each input record simply gets weight `1`:


In [2]:
tn = Tn.source(click_source_str).to_zSet(Tn.from_records)._peek().sink(sink_str)
#
built_tn = Tn.build(tn)
#
m_list = click_generator.generate(2)

print("Input records:")
for m in m_list:
    print(m)

print("\nZ-set:")
_ = built_tn.process({click_source_str: m_list})


Input records:
{'key': None, 'value': {'customer_id': 10, 'view_time': 26, 'ts': 1787403043934}}
{'key': None, 'value': {'customer_id': 63, 'view_time': 60, 'ts': 1787403044034}}

Z-set:
({'key': None, 'value': {'customer_id': 10, 'view_time': 26, 'ts': 1787403043934}}, 1)
({'key': None, 'value': {'customer_id': 63, 'view_time': 60, 'ts': 1787403044034}}, 1)


<a id="from_debezium"></a>
##### from_debezium

Append-only streams from Kafka can also include weights in the sense of DBSP, e.g. if using the *Debezium* format where each record is marked by an operation type in the `op` field of the `value` field.

Here is an example where we generate two input records using the Debezium format:
1. a *change* (`"op": "c"`) that gets weight `1` in pydbsp
2. a *delete* (`"op": "d"`) that gets weight `-1` in pydbsp


In [3]:
tn = Tn.source(click_source_str).to_zSet(Tn.from_debezium)._peek().sink(sink_str)
#
built_tn = Tn.build(tn)
#
m_list = debezium_click_generator.generate(1, w=1) + debezium_click_generator.generate(1, w=-1)

print("Input records:")
for m in m_list:
    print(m)

print("\nZ-set:")
_ = built_tn.process({click_source_str: m_list})


Input records:
{'key': None, 'value': {'customer_id': 73, 'view_time': 117, 'ts': 1787403043934, 'before': None, 'after': {'customer_id': 73, 'view_time': 117, 'ts': 1787403043934}, 'op': 'c'}}
{'key': None, 'value': {'customer_id': 72, 'view_time': 40, 'ts': 1787403044034, 'before': {'customer_id': 72, 'view_time': 40, 'ts': 1787403044034}, 'after': None, 'op': 'd'}}

Z-set:
({'key': None, 'value': {'customer_id': 73, 'view_time': 117, 'ts': 1787403043934}}, 1)
({'key': None, 'value': {'customer_id': 72, 'view_time': 40, 'ts': 1787403044034}}, -1)


<a id="_from_records"></a>
##### _from_records

This option allows you to explicitly specify the weights of the incoming records.

Here is an example:


In [4]:
tn = Tn.source(click_source_str).to_zSet(Tn._from_records)._peek().sink(sink_str)
#
built_tn = Tn.build(tn)
#
m_w_tuple_list = weights_click_generator.generate(1, w=1) + weights_click_generator.generate(1, w=-1)

print("Input record/weight tuples:")
for m_w_tuple in m_w_tuple_list:
    print(m_w_tuple)

print("\nZ-set:")
_ = built_tn.process({click_source_str: m_w_tuple_list})


Input record/weight tuples:
({'key': None, 'value': {'customer_id': 50, 'view_time': 47, 'ts': 1787403043934}}, 1)
({'key': None, 'value': {'customer_id': 71, 'view_time': 61, 'ts': 1787403044034}}, -1)

Z-set:
({'key': None, 'value': {'customer_id': 50, 'view_time': 47, 'ts': 1787403043934}}, 1)
({'key': None, 'value': {'customer_id': 71, 'view_time': 61, 'ts': 1787403044034}}, -1)


<a id="sinks"></a>
### Sinks

You specify sinks using the [`sink()`](#source) operator.

With the [`from_ZSet`](#from_zSet) operator, you can configure how Kafi Streams converts the *Z-sets* returned from the pydbsp processing steps are converted into output records-


<a id="build"></a>
### build()

Central method for building a topology, i.e., wiring it up with the underlying pydbsp library:

```    
@staticmethod
def build(*sink_tn_tuple):
    """Build the circuit for one or more sink nodes.
    
    Args:
        *sink_tn_tuple: one or more sink tn to build
    Returns:
        built_tn: the built topology node"""
```


<a id="reset"></a>
### reset()

With `reset()`, you can rebuild the pydbsp circuit attached to a built topology node from scratch. This also clears all state:

```
def reset(self):
    """Rebuild the circuit from scratch,
    clearing all state."""
```

---
<a id="run_debug"></a>
## Run and debug

This is about running How can I run a Streams thread?

<a id="start_streams"></a>
### start_streams()

Start a Streams thread and get the `stop_fun: None -> None` to stop it:

```
@staticmethod
def start_streams(built_tn, checkpoint_storage=None, checkpoint_topic=None, checkpoint_interval=default_checkpoint_interval_float, **kwargs):
"""Run streams() in a background thread; returns a function to stop it.

Args:
    built_tn: built tn to run
    checkpoint_storage: storage backend for checkpoints, or None to disable checkpointing
    checkpoint_topic: topic name used to store checkpoints
    checkpoint_interval: seconds between checkpoints
    **kwargs: passed through to streams()
Returns:
    stop_fun: None -> None function to stop the Streams processing thread"""
```


<a id="streams"></a>
### streams()

Synchronous sub method for `start_streams` - can e.g. be used to run Streams synchronously:

```
@staticmethod
def streams(built_tn, checkpoint_storage=None, checkpoint_topic=None, checkpoint_interval=default_checkpoint_interval_float, stop_event=None, **kwargs):
"""Build producers/consumers from the topology's sources/sinks and run streams_fun().

Args:
    built_tn: built tn to run
    checkpoint_storage: storage backend for checkpoints, or None to disable checkpointing
    checkpoint_topic: topic name used to store checkpoints
    checkpoint_interval: seconds between checkpoints
    stop_event: threading.Event that stops the loop once set
    **kwargs: passed through to storage.producer()/consumer()"""
```


<a id="streams_fun"></a>
### streams_fun()

Sub method for `streams()`, actually the main method of Streams. Can be called directly e.g. if you'd like to set up the consumers/producers yourself:

```
@staticmethod
def streams_fun(built_tn, sink_str_foreach_fun_finally_fun_tuple_dict, checkpoint_storage=None, checkpoint_topic=None, checkpoint_interval=default_checkpoint_interval_float, stop_event=None, **kwargs):
    """Main streams loop: consume, push through the topology, produce, checkpoint, repeat.

    Args:
        built_tn: built tn to run
        sink_str_foreach_fun_finally_fun_tuple_dict: dict, sink_str -> (produce_fun, close_fun)
        checkpoint_storage: storage backend for checkpoints, or None to disable checkpointing
        checkpoint_topic: topic name used to store checkpoints
        checkpoint_interval: seconds between checkpoints
        stop_event: threading.Event that stops the loop once set
        **kwargs: extra options, e.g. group, step_fun, progress, chunk_size_bytes"""
```


<a id="threads"></a>
### threads()

```
@staticmethod
def threads():
    """All currently running Streams background threads.
    
    Returns:
        streams_thread_list: the list of currently running Streams threads"""
```


---
<a id="stop"></a>
## Stop

This is about running How can I run a Streams thread?